# 엑셀 데이터와 PDF 문서를 모두 활용하는 챗봇

## 실습 목표
---
다양한 기능을 한데 모아 엑셀 데이터와 인공지능 산업 동향 연구 문서를 모두 활용할 수 있는 챗봇을 구성하고 사용해봅니다.
- 엑셀 데이터로 데이터 분석 코드를 실행하고 답변하는 기능
- 엑셀 데이터로 데이터 분석 그래프를 그리고 저장하는 기능
- 인공지능 산업 동향 연구 문서 기반 QA 챗봇

## 실습 목차
---

1. **각 기능 별 그래프 노드 정의:** 추가하고자 하는 다양한 기능에 대응하는 노드를 정의합니다.

2. **챗봇 고도화:** LangGraph를 활용해서 엑셀 데이터와 인공지능 산업 동향 연구 문서를 모두 활용할 수 있는 챗봇을 구성합니다.

## 0. 환경 설정
- 필요한 라이브러리를 불러옵니다.

In [ ]:
import contextlib
import io
import os
import time
from typing import Any, Dict, List, Optional, Tuple

import pandas as pd
from IPython.display import Image, display
from langchain.document_loaders import PyPDFLoader
from langchain_chroma import Chroma
from langchain_community.vectorstores import FAISS
from langchain_core.output_parsers import JsonOutputParser, StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_ollama import ChatOllama, OllamaEmbeddings
from langgraph.graph import END, StateGraph
from typing_extensions import TypedDict

- Ollama를 통해 llama 3.1 8B 모델을 불러옵니다.

In [ ]:
!ollama pull llama3.1

llama 3.1 8B 모델을 사용하는 ChatOllama 객체를 생성하고, 엑셀 데이터를 불러옵니다.
- 출처: 한국지능정보사회진흥원_인공지능 학습용 데이터 구축 현황
  - https://www.data.go.kr/data/15039915/fileData.do

In [ ]:
llm = ChatOllama(model="llama3.1")
route_llm = ChatOllama(model="llama3.1", format="json")

excel_data_name = "한국지능정보사회진흥원_인공지능 학습용 데이터 구축 현황_20210104.csv"
pdf_data_name = "RE177_2023년 국내외 인공지능 산업 동향 연구_2장.pdf"

# 데이터를 불러오고, 이름과 컬럼명을 저장합니다.
data_dir = "./data"
df_ai_train_data_dist = pd.read_csv(
    os.path.join(data_dir, excel_data_name), index_col=None
)

# 데이터를 저장한 변수명을 LLM에 제공하여 이 변수를 활용하는 코드를 작성하게 할 수 있습니다.
df_name = "df_ai_train_data_dist"
df_columns = ", ".join(df_ai_train_data_dist.columns)

인공지능 산업 동향 연구 문서를 불러와서 `OllamaEmbeddings`를 활용해 벡터로 변환하고, Chroma DB 혹은 FAISS DB를 활용하여 저장합니다.
- 출처: 소프트웨어정책연구소 (SRPI) 2023년 국내외 인공지능 산업 동향 연구 보고서 (일부 발췌)
  - 본 과정에서는 전체 보고서 중 제2장 인공지능 산업 현황 및 전망 구간만 발췌하여 사용합니다.
  - https://spri.kr/posts/view/23728?code=research&study_type=&board_type=&flg=#none

- 벡터 변환 및 저장 과정은 약 1분 정도 소요됩니다.

In [ ]:
%%time
embeddings = OllamaEmbeddings(model="llama3.1")

# 시장 조사 문건을 불러옵니다.
doc_path = os.path.join(data_dir, pdf_data_name)
loader = PyPDFLoader(doc_path)
docs = loader.load()

# FAISS 또는 Chroma를 사용하여 문서를 벡터화합니다.
# vectorstore_faiss = FAISS.from_documents(
#     docs,
#     embedding=embeddings
# )

vectorstore = Chroma.from_documents(
    docs,
    embedding=embeddings
)

db_retriever = vectorstore.as_retriever()

## 1. 각 기능 별 그래프 노드 정의
- 추가하고자 하는 다양한 기능에 대응하는 노드를 정의합니다.

이전 실습에서 사용한 노드와 관련 함수를 먼저 정의합니다.

In [ ]:
# LLM이 생성한 코드를 파싱하는 함수를 정의합니다.
def python_code_parser(input: str) -> str:
    # LLM은 대부분 ``` 블럭 안에 코드를 출력합니다. 이를 활용합니다.
    # ```python (코드) ```, 혹은 ``` (코드) ``` 형태로 출력됩니다. 두 경우 모두에 대응하도록 코드를 작성합니다.
    processed_input = input.replace("```python", "```").strip()
    parsed_input_list = processed_input.split("```")

    # 만약 ``` 블럭이 없다면, 입력 텍스트 전체가 코드라고 간주합니다.
    # 아닐 경우 이어지는 코드 실행 과정에서 예외 처리를 통해 오류를 확인할 수 있습니다.
    if len(parsed_input_list) == 1:
        return processed_input

    # 코드 부분만 추출합니다.
    # LLM은 여러 코드 블럭에 걸쳐 필요한 코드를 출력할 수 있으므로, 코드가 있는 홀수 번째 텍스트를 모두 저장합니다.
    parsed_code_list = []
    for i in range(1, len(parsed_input_list), 2):
        parsed_code_list.append(parsed_input_list[i])

    # 코드 부분을 하나로 합칩니다.
    return "\n".join(parsed_code_list)


# 생성한 코드를 실행하는 함수를 정의합니다.
def run_code(input_code: str):
    # 코드가 출력한 값을 캡쳐하기 위한 StringIO 객체를 생성합니다.
    output = io.StringIO()
    try:
        # Redirect stdout to the StringIO object
        with contextlib.redirect_stdout(output):
            # 코드가 실행하면서 출력한 모든 결과를 캡쳐합니다.
            exec(input_code, {"df_ai_train_data_dist": df_ai_train_data_dist})
    except Exception as e:
        # 에러가 발생할 경우, 이를 StringIO 객체에 저장합니다.
        print(f"Error: {e}", file=output)
    # StringIO 객체에 저장된 값을 반환합니다.
    return output.getvalue()

In [ ]:
class State(TypedDict):
    # 그래프 상태의 속성을 정의합니다.
    # 질문, LLM이 생성한 텍스트, 데이터, 코드를 저장합니다.
    question: str
    generation: str
    data: str
    code: str


# 그래프를 구성하기 위해 StateGraph 객체를 생성합니다.
# 생성자의 인자로 State를 전달하여 Node 간에 정보를 전달할 때 State type을 사용함을 명시합니다.
workflow = StateGraph(State)

그래프의 Node는 체인의 각 구성 요소에 대응합니다. Agent, Tool, LLM 등 그래프의 각 구성 요소를 의미합니다.

Edge는 Node를 연결하는 요소로, Node에서 정보를 어느 Node로 전달해야 하는지를 나타냅니다.
- 체인은 일렬로 이어져 있기 때문에, 사용자의 입력을 연결된 순서대로 통과시키면 원하는 결과를 얻을 수 있습니다.
- 하지만, 그래프는 일렬로 이어져 있지 않기 때문에 Edge를 통해 정보를 전달하는 순서 및 방향을 정해줘야 합니다.

In [ ]:
## Node 생성
# Node는 그래프에서 실행될 수 있는 작업을 정의합니다.
# Node는 함수로 정의되며, StateGraph를 정의할 때 사용한 State type을 입력으로 받습니다.
# Node는 state를 업데이트하거나, 새로운 state를 반환할 수 있습니다.


def query(state: State):
    """
    데이터를 쿼리하는 코드를 생성하고, 실행하고, 그 결과를 포함한 State를 반환합니다.
    위 과정은 앞서 정의한 `find_data` 함수를 활용합니다.

    Args:
        state (dict): 현재 그래프 상태

    Returns:
        state (dict): 쿼리한 데이터를 포함한 새로운 State
    """
    print("---데이터 쿼리---")  # 현재 상태를 확인하기 위한 Print문
    question = state["question"]

    # Retrieval
    system_message = "당신은 주어진 데이터를 분석하는 데이터 분석가입니다.\n"
    system_message += f"주어진 DataFrame에서 데이터를 출력하여 주어진 질문에 답할 수 있는 파이썬 코드를 작성하세요. "
    system_message += f"{df_name} DataFrame에 액세스할 수 있습니다.\n"
    system_message += (
        f"`{df_name}` DataFrame에는 다음과 같은 열이 있습니다: {df_columns}\n"
    )
    system_message += (
        "데이터는 이미 로드되어 있으므로 데이터 로드 코드를 생략해야 합니다."
    )

    message_with_data_info = [
        ("system", system_message),
        ("human", "{question}"),
    ]

    prompt_with_data_info = ChatPromptTemplate.from_messages(message_with_data_info)

    # 체인을 구성합니다.
    code_generate_chain = (
        {"question": RunnablePassthrough()}
        | prompt_with_data_info
        | llm
        | StrOutputParser()
        | python_code_parser
    )
    code = code_generate_chain.invoke(question)
    data = run_code(code)
    return {"question": question, "code": code, "data": data, "generation": code}


def answer_with_data(state: State):
    """
    쿼리한 데이터를 바탕으로 답변을 생성합니다.

    Args:
        state (dict): 현재 그래프 상태

    Returns:
        state (dict): LLM의 답변을 포함한 새로운 State
    """
    print("---데이터 기반 답변 생성---")  # 현재 상태를 확인하기 위한 Print문
    question = state["question"]
    data = state["data"]

    # 데이터를 바탕으로 질문에 대답하는 코드를 생성합니다.
    reasoning_system_message = (
        "당신은 데이터를 바탕으로 질문에 답하는 데이터 분석가입니다.\n"
    )
    reasoning_system_message += f"사용자가 입력한 데이터를 바탕으로, 질문에 대답하세요."

    reasoning_user_message = "데이터: {data}\n{question}"

    reasoning_with_data = [
        ("system", reasoning_system_message),
        ("human", reasoning_user_message),
    ]
    reasoning_with_data_chain = (
        ChatPromptTemplate.from_messages(reasoning_with_data) | llm | StrOutputParser()
    )

    # 대답 생성
    generation = reasoning_with_data_chain.invoke({"data": data, "question": question})
    return {
        "question": question,
        "code": state["code"],
        "data": data,
        "generation": generation,
    }


def answer(state: State):
    """
    데이터를 쿼리하지 않고 답변을 바로 생성합니다.

    Args:
        state (dict): 현재 그래프 상태

    Returns:
        state (dict): LLM의 답변을 포함한 새로운 State
    """
    print("---답변 생성---")  # 현재 상태를 확인하기 위한 Print문
    question = state["question"]

    return {"question": question, "generation": llm.invoke(question).content}

이번 실습에서 추가할 기능에 대응하는 Node를 구현해 보겠습니다.
- 1. `plot_graph`: 그래프를 Plot하는 노드
- 2. `retrival`: RAG를 적용하는 노드
- 3. `answer_with_retrieved_data`: RAG 결과를 바탕으로 답변하는 노드

In [ ]:
def plot_graph(state: State):
    """
    현재 그래프 상태를 시각화합니다.

    Args:
        state (dict): 현재 그래프 상태

    Returns:
        None
    """
    print("---그래프 시각화---")  # 현재 상태를 확인하기 위한 Print문
    question = state["question"]

    # Draw Graph
    system_message = "당신은 주어진 데이터를 분석하는 데이터 분석가입니다.\n"
    system_message += f"주어진 DataFrame에서 데이터를 추출하여 사용자의 질문에 답할 수 있는 그래프를 그리는 코드를 작성하세요. "
    system_message += f"{df_name} DataFrame에 액세스할 수 있습니다.\n"
    system_message += (
        f"`{df_name}` DataFrame에는 다음과 같은 열이 있습니다: {df_columns}\n"
    )
    system_message += (
        "데이터는 이미 로드되어 있으므로 데이터 로드 코드를 생략해야 합니다."
    )

    message_with_data_info = [
        ("system", system_message),
        ("human", "{question}"),
    ]

    prompt_with_data_info = ChatPromptTemplate.from_messages(message_with_data_info)

    # 체인을 구성합니다.
    code_generate_chain = (
        {"question": RunnablePassthrough()}
        | prompt_with_data_info
        | llm
        | StrOutputParser()
        | python_code_parser
    )
    code = code_generate_chain.invoke(question)

    if "plt.show()" in code:
        code = code.replace("plt.show()", 'plt.savefig("plot.png")')
    answer = run_code(code)
    return {"question": question, "code": code, "data": answer, "generation": code}


def retrieval(state: State):
    """
    데이터 검색을 수행합니다.

    Args:
        state (dict): 현재 그래프 상태

    Returns:
        state (dict): 검색된 데이터를 포함한 새로운 State
    """

    def get_retrieved_text(docs):
        result = "\n".join([doc.page_content for doc in docs])
        return result

    print("---데이터 검색---")  # 현재 상태를 확인하기 위한 Print문
    question = state["question"]

    # Retrieval Chain
    retrieval_chain = db_retriever | get_retrieved_text

    data = retrieval_chain.invoke(question)

    return {"question": question, "data": data}


def answer_with_retrieved_data(state: State):
    """
    검색된 데이터를 바탕으로 답변을 생성합니다.

    Args:
        state (dict): 현재 그래프 상태

    Returns:
        state (dict): LLM의 답변을 포함한 새로운 State
    """

    print(
        "---검색된 데이터를 바탕으로 답변 생성---"
    )  # 현재 상태를 확인하기 위한 Print문

    question = state["question"]
    data = state["data"]

    messages_with_contexts = [
        ("system", "사용자가 입력하는 정보를 바탕으로 질문에 답하세요."),
        ("human", "정보: {context}.\n{question}."),
    ]
    prompt_with_context = ChatPromptTemplate.from_messages(messages_with_contexts)

    # 체인 구성
    qa_chain = prompt_with_context | llm | StrOutputParser()

    generation = qa_chain.invoke({"context": data, "question": question})
    return {"question": question, "data": data, "generation": generation}

**라우팅 전문가 페르소나**를 적용한 체인을 통해 원하는 로직을 선택할 수 있도록 설정해봅시다. 

In [ ]:
# 시스템 메시지에 사용 가능한 툴과 각 툴을 사용할 상황을 명시합니다.
# 수월한 선택을 위해 JSON 형식으로 출력하도록 프롬프트에 지정합니다.
route_system_message = """당신은 사용자의 질문에 RAG, 엑셀 데이터 중 어떤 것을 활용할 수 있는지 결정하는 전문가입니다.
인공지능 산업 동향과 관련된 질문이라면 RAG를 활용하세요.
인공지능 데이터 프로필과 관련된 질문이라면 excel_data를 활용하세요.
그래프를 그리라는 질문이면 excel_plot을 활용하세요.
둘 다 아니라면, plain_answer로 충분합니다.
주어진 질문에 맞춰 `rag`, `excel_data`, `excel_plot`, `plain_answer`중 하나를 선택하세요.
답변은 `route` key 하나만 있는 JSON으로 답변하고, 다른 텍스트나 설명을 생성하지 마세요."""
route_user_message = "{question}"
route_prompt = ChatPromptTemplate.from_messages(
    [("system", route_system_message), ("human", route_user_message)]
)

# 로직 선택용 ChatOllama 객체를 생성합니다. format="json" 인자를 적용하여 출력 양식을 json으로 강제합니다.
# 같은 질문에 항상 같은 대답을 유도하기 위해 temperature를 0으로 설정합니다.
route_llm = ChatOllama(model="llama3.1", format="json", temperature=0)
router_chain = route_prompt | route_llm | JsonOutputParser()

다양한 질문에 대해 테스트 해보고, 그 결과를 확인해 봅시다.

In [ ]:
print(router_chain.invoke({"question": "인공지능 산업 현황 및 전망에 대해 알려줘"}))
print(router_chain.invoke({"question": "인공지능 학습 데이터의 종류를 요약해줘"}))
print(router_chain.invoke({"question": "오늘 저녁 뭐 먹을까?"}))
print(
    router_chain.invoke(
        {"question": "인공지능 학습 데이터의 년도별 개수를 그래프로 그려줘"}
    )
)

대부분 잘 결정하는 것을 볼 수 있습니다.<br>
이 체인을 활용해서 고도화된 `init_answer` 함수를 정의합니다.

In [ ]:
def init_answer(state: State) -> str:
    "초기 질문의 경로를 결정합니다."
    question = state["question"]
    route = router_chain.invoke({"question": question})["route"]
    return {"question": question, "generation": route}


def route_question(state: State) -> str:
    route = state["generation"]
    return route.lower().strip()

지난 실습과 달리 총 4개의 노드를 사용하고, 조건에 따라 사용하는 노드가 달라집니다.

이를 구현하기 위해, 노드와 간선을 그래프에 추가합니다.

In [ ]:
## 그래프 구성

# 앞서 정의한 Node를 모두 추가합니다.
workflow.add_node("init_answer", init_answer)

workflow.add_node("excel_data", query)
workflow.add_node("rag", retrieval)

workflow.add_node("excel_plot", plot_graph)
workflow.add_node("answer_with_data", answer_with_data)
workflow.add_node("plain_answer", answer)
workflow.add_node("answer_with_retrieval", answer_with_retrieved_data)

# 시작지점을 정의합니다.
workflow.set_entry_point("init_answer")

# 간선을 정의합니다.
# END는 종결 지점을 의미합니다.
workflow.add_edge(
    "plain_answer", END
)  # workflow.set_finish_point("answer")와 동일합니다.
workflow.add_edge("answer_with_data", END)
workflow.add_edge("answer_with_retrieval", END)
workflow.add_edge("excel_plot", END)  # 그래프를 그리고 종결합니다.
workflow.add_edge("excel_data", "answer_with_data")
workflow.add_edge("rag", "answer_with_retrieval")


# 조건부 간선을 정의합니다.
# init_answer 노드의 답변을 바탕으로 decide_query 함수에서 query 또는 answer로 분기합니다.
workflow.add_conditional_edges(
    "init_answer",
    route_question,
    # 어떤 노드로 이동할지 mapping합니다. 없어도 무방하지만, Graph의 가독성을 높일 수 있습니다.
    {
        "excel_data": "excel_data",
        "rag": "rag",
        "excel_plot": "excel_plot",
        "plain_answer": "plain_answer",
    },
)

Node, Edge, 분기를 모두 구성했으니 이제 그래프를 컴파일 하고, 그 구조를 확인해 봅시다.

In [ ]:
graph = workflow.compile()
display(Image(graph.get_graph().draw_mermaid_png()))

이제 챗봇을 사용해봅시다.<br>
다양한 종류의 데이터가 로드된 만큼, 어떤 데이터에 대한 요청인지 프롬프트에 명시하는 것이 좋습니다.

- 예시 질문 (인공지능 산업 동향 연구 문서 활용): 인공지능 산업 현황 및 전망에 대해 알려줘
- 예시 질문 (데이터 활용): 인공지능 학습 데이터의 종류를 요약해줘
- 예시 질문 (그래프): 인공지능 학습 데이터의 년도별 개수를 그래프로 그려줘
- 예시 질문 (데이터 무관): 오늘 저녁 뭐 먹을까?

답변 생성에는 평균 20~30초 정도 소요됩니다. 1분 넘게 답변이 생성되지 않을 경우 강제 종료 후 재시작 해주세요.

In [ ]:
while True:
    question = input("질문을 입력해주세요 (종료를 원하시면 '종료'를 입력해주세요.): ")
    if question == "종료":
        break
    else:
        # graph.invoke 함수를 사용하여 그래프를 실행하고, 최종 결과를 반환합니다.
        # 답변 생성에는 약 1분 정도 소요됩니다.
        response = graph.invoke({"question": question})
        if "code" in response and "plt.savefig" in response["code"]:
            display(Image("plot.png"))
        else:
            print("Assistant: ", response["generation"])

Vector DB를 추출하여 저장합니다.

In [ ]:
# ChromaDB는 Vector DB를 생성할 때 persist_directory인자를 지정하면 해당 디렉토리에 저장됩니다.

vectorstore = Chroma.from_documents(
    docs,
    embedding=embeddings,
    persist_directory="./vectorstore/chroma"
)

# FAISS는 save_local 메서드를 사용하여 저장할 수 있습니다.
# vectorstore_faiss.save_local("./vectorstore/faiss")